In [2]:
%load_ext autoreload
%autoreload 2

import os
os.chdir('/home/xiaowenz/finetune')

In [3]:
from PIL import Image
from pathlib import Path
import datasets

def convert_to_swift_format(conversation, media_dir):
  """
  Convert a conversation from the current format to Swift format.

  Args:
      conversation: List of message dictionaries in the format:
          [{'role': 'system', 'content': '...'},
           {'role': 'user', 'content': [{'video': 'path/to/video.mp4'}]},
           {'role': 'user', 'content': 'Question text'},
           {'role': 'assistant', 'content': 'Answer tdext'}]

  Returns:
      Dictionary in Swift format with 'messages' key and optional media keys
  """
  result = {"messages": []}
  images = []
  videos = []
  audios = []

  # Track media placeholders for content substitution
  image_count = 0
  video_count = 0
  audio_count = 0

  for message in conversation:
    role = message['role']
    content = message['content']

    # Process content based on its type
    if isinstance(content, str):
      # Simple text content
      messages = result['messages']
      if messages and messages[-1]['role'] == role:
        messages[-1]['content'] += '\n' + content
      else:
        result["messages"].append({
            "role": role,
            "content": content
        })
    elif isinstance(content, list):
      # Content with potential media
      text_parts = []

      for item in content:
        if isinstance(item, dict):
          if 'text' in item:
            text_parts.append(item['text'])
          elif 'image' in item:
            text_parts.append("<image>")
            img = item['image']
            if isinstance(img, Image.Image):
              # Save the image to the media directory
              image_path = media_dir / f"image_{image_count}.png"
              img.save(image_path)
              img = image_path
            images.append(img)
            image_count += 1
          elif 'video' in item:
            text_parts.append("<video>")
            videos.append(item['video'])
            video_count += 1
          else:
            raise ValueError(f"Unsupported media type in item: {item}")
        elif isinstance(item, str):
          text_parts.append(item)

      # Join all text parts
      content_text = ''.join(text_parts)
      messages = result['messages']
      if messages and messages[-1]['role'] == role:
        messages[-1]['content'] += '\n' + content_text
      else:
        result["messages"].append({
            "role": role,
            "content": content_text
        })

  # Add media arrays if they exist
  if images:
    result["images"] = images
  if videos:
    result["videos"] = videos
  if audios:
    result["audios"] = audios

  return result


def convert_conversations_to_swift_format(conversations: list[dict], media_dir: str | Path) -> list[dict]:
  """
  Convert a list of conversations to Swift format.

  Args:
      conversations: List of conversations

  Returns:
      List of dictionaries in Swift format
  """
  n = len(conversations)
  z_fill = lambda x: str(x).zfill(len(str(n)))
  media_dir = Path(media_dir)
  return [
    convert_to_swift_format(conv, media_dir / z_fill(i))
    for i, conv in enumerate(conversations)
  ]


def convert_datasetdict(conv_dict: dict[str, list[dict]], media_dir: str | Path) -> dict[str, list[dict]]:
  """
  Convert a dataset dictionary to Swift format.

  Args:
      conv_dict: Dictionary with keys as conversation IDs and values as lists of messages

  Returns:
      Dictionary in Swift format
  """
  media_dir = Path(media_dir)
  return {
      k: convert_conversations_to_swift_format(v, media_dir / k)
      for k, v in conv_dict.items()
  }

In [15]:
from qwenvl.data import avail_datasets
from qwenvl.data.conversation import *
from qwenvl.data.preprocess import VerifyMediaStrategy
from qwenvl.data.prompts import SYS_PROMPTS, USR_PROMPTS
from qwenvl.utils import get_logger

logger = get_logger(__name__)
def create_swift_dataset(
    ds_name: str,
    split: str,
    modifiers: list[ConversationModifier] = [],
    **kwargs
):
  ds_config = avail_datasets[ds_name]
  print(ds_config)
  ds = datasets.load_dataset(ds_config['ds_key'], split=split)
  print(ds)
  ids = [i for i in range(len(ds))]
  base_cm = ds_config['cm'](
      **ds_config,
      **kwargs
  )
  
  cp = ConversationProcessor(
      conversation_maker=base_cm,
      conversation_modifiers=modifiers,
      **ds_config
  )
  
  ds = VerifyMediaStrategy(cp.get_content, None)(ds)
  convos = [cp(item) for item in ds]
  answers = ds['answer']
  logger.info(f"Processed {len(convos)} items in split '{split}'")

  idx = 0
  logger.info(f"Pre conversion: {convos[idx]}")

  ds = convert_conversations_to_swift_format(convos, ds_config['media_dir'])
  ds = datasets.DatasetDict({
    'train': datasets.Dataset.from_list(ds).add_column('label', answers).add_column('id', ids).shuffle(),
  })

  logger.info(f"Post conversion: {ds['train'][idx]}")
  return ds

In [16]:
ds_name = 'surgeryvid_tiny'
ds = create_swift_dataset(ds_name, split='test', for_training=False);
ds.push_to_hub(f"withcomment/swift_{ds_name}")

{'ds_dir': '/scratch/xiaowenz/datasets/surgeryvid_tiny/data/dataset', 'media_dir': '/scratch/xiaowenz/datasets/surgeryvid/data/vid_processed', 'ds_key': 'withcomment/surgeryvid_tiny', 'cm': <class 'qwenvl.data.conversation.VQACM'>}
Dataset({
    features: ['video_id', 'video_url', 'video', 'timestamp', 'question', 'answer', 'video_metadata', 'num_media', 'num_media_tokens', 'num_tokens'],
    num_rows: 154
})
2025-08-09 11:10:01,211 - __main__ - INFO - Processed 154 items in split 'test'
2025-08-09 11:10:01,212 - __main__ - INFO - Pre conversion: [{'role': 'user', 'content': [{'video': '/scratch/xiaowenz/datasets/surgeryvid/data/vid_processed/MHXVLjFxDGE_570_612.mp4'}]}, {'role': 'user', 'content': 'What anatomical area contains sentinel lymph nodes being examined?'}]
2025-08-09 11:10:01,226 - __main__ - INFO - Post conversion: {'messages': [{'content': '<video>\nWhat type of surgical procedure is demonstrated?', 'role': 'user'}], 'videos': ['/scratch/xiaowenz/datasets/surgeryvid/data/

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/withcomment/swift_surgeryvid_tiny/commit/693e99f1e95ae89301b837263b44f7b3266eb421', commit_message='Upload dataset', commit_description='', oid='693e99f1e95ae89301b837263b44f7b3266eb421', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/withcomment/swift_surgeryvid_tiny', endpoint='https://huggingface.co', repo_type='dataset', repo_id='withcomment/swift_surgeryvid_tiny'), pr_revision=None, pr_num=None)

In [ ]:
ds = create_swift_dataset('openbiomedvid_qa', split='train', for_training=True);
ds.push_to_hub(f"withcomment/{ds_fullname}_train")